In [1]:
import numpy as np
import nibabel as nb
import Functional_Fusion.dataset as fdata
import cortico_cereb_connectivity.globals as gl
import cortico_cereb_connectivity.summarize as cs
import cortico_cereb_connectivity.scripts.script_plotting_utils as plu
import matplotlib.pyplot as plt

In [ ]:
traindata = gl.traindata_string(None)
cortex_roi = "Icosahedron1002"
cerebellum_roi = "NettekovenSym32"
# method = 'L2reg';       ex = 'A4_global'
method = 'NNLS';        ex = 'A1_global'

fname = gl.conn_dir + f'/maps/{traindata}_{cortex_roi}_{method}_{cerebellum_roi}.pscalar.nii'
oname = f'{traindata}_{cortex_roi}_{method}_{cerebellum_roi}.dscalar.nii'

In [9]:
cifti_img = cs.avrg_weight_map_roi(traindata=traindata,
                                   cortex_roi=cortex_roi,
                                   method=method,
                                   extension=ex,
                                   cerebellum_roi=cerebellum_roi,
                                   norm=True,
                                   ynorm=True)
cifti_img_new = cs.sort_roi_rows(cifti_img)
nb.save(cifti_img_new, fname)

In [10]:
cifti_img_new = nb.load(fname)
labels = cifti_img_new.header.get_axis(0).name
# index,colors,labels = nt.read_lut(gl.atlas_dir + f"/tpl-MNI152NLin2009cSymC/atl-{cerebellum_roi}.lut")

In [11]:
# smooth the cifti
cs.pscalar_to_smoothed_dscalar(cifti_img_new, oname, sigma=2.0)

wb_command -cifti-smoothing 'temp.dscalar.nii' 2.0 0.0 COLUMN /cifs/diedrichsen/data/Cerebellum/connectivity/maps/MdWfIbDeHtNiSoScLa_Icosahedron1002_NettekovenSym32_NNLS_mean.dscalar.nii -left-surface /cifs/diedrichsen/data/FunctionalFusion_new/Atlases/tpl-fs32k/tpl-fs32k_hemi-L_inflated.surf.gii -right-surface /cifs/diedrichsen/data/FunctionalFusion_new/Atlases/tpl-fs32k/tpl-fs32k_hemi-R_inflated.surf.gii


In [10]:
def make_cifti_subset(cifti_img, region_index):
    data = cifti_img.get_fdata()
    
    # Keep the selected row as (1, n_brainordinates)
    row = data[region_index:region_index + 1, :]

    # Get the two CIFTI axes
    axis0 = cifti_img.header.get_axis(0)
    axis1 = cifti_img.header.get_axis(1)

    # Subset only the first axis
    new_axis0 = axis0[region_index:region_index + 1]

    # Construct a header with the correct axes
    header = nb.cifti2.Cifti2Header.from_axes(
        (new_axis0, axis1)
    )

    img = nb.Cifti2Image(
        row,
        header=header,
        nifti_header=cifti_img.nifti_header
    )

    return img

In [ ]:
# Smooth the connectivity maps for display 
# Question: Do we need this since we have the overall connectivity weights? 
for i, label in enumerate(labels):
    print(f"Label {i+1}: {label}")
    cifti_img_region = make_cifti_subset(cifti_img_new, i)
    # make the smoothed version of that map
    oname = f'{traindata}_{cortex_roi}_{label}_{method}_mean.dscalar.nii'
    cs.pscalar_to_smoothed_dscalar(cifti_img_region, oname, sigma=2.0)

Label 1: 0
wb_command -cifti-smoothing 'temp.dscalar.nii' 2.0 0.0 COLUMN /cifs/diedrichsen/data/Cerebellum/connectivity/maps/MdWfIbDeHtNiSoScLa_Icosahedron1002_0_L2reg_mean.dscalar.nii -left-surface /cifs/diedrichsen/data/FunctionalFusion_new/Atlases/tpl-fs32k/tpl-fs32k_hemi-L_inflated.surf.gii -right-surface /cifs/diedrichsen/data/FunctionalFusion_new/Atlases/tpl-fs32k/tpl-fs32k_hemi-R_inflated.surf.gii
Label 2: 1
wb_command -cifti-smoothing 'temp.dscalar.nii' 2.0 0.0 COLUMN /cifs/diedrichsen/data/Cerebellum/connectivity/maps/MdWfIbDeHtNiSoScLa_Icosahedron1002_1_L2reg_mean.dscalar.nii -left-surface /cifs/diedrichsen/data/FunctionalFusion_new/Atlases/tpl-fs32k/tpl-fs32k_hemi-L_inflated.surf.gii -right-surface /cifs/diedrichsen/data/FunctionalFusion_new/Atlases/tpl-fs32k/tpl-fs32k_hemi-R_inflated.surf.gii
Label 3: 10
wb_command -cifti-smoothing 'temp.dscalar.nii' 2.0 0.0 COLUMN /cifs/diedrichsen/data/Cerebellum/connectivity/maps/MdWfIbDeHtNiSoScLa_Icosahedron1002_10_L2reg_mean.dscalar.n

## Investigate the connectivity weights based on Glasser ROIs 

In [ ]:
weights_file = gl.conn_dir + '/maps/MdWfIbDeHtNiSoScLa_Icosahedron1002_NNLS_NettekovenSym32.pscalar.nii'
label_file = gl.conn_dir + '/maps/glasser_Icosahedron1002.plabel.nii'
T = cs.make_weight_table(weights_file,label_file)

,cortical_roi,row_num,row_name,length,mean,max
0,???,0,A1L,8,0.011345,0.041464
1,???,1,A1R,8,0.013063,0.033557
2,???,2,A2L,8,0.005992,0.027677
3,???,3,A2R,8,0.006402,0.036888
4,???,4,A3L,8,0.013082,0.077594
...,...,...,...,...,...,...
5787,L_p24_ROI,27,S3R,6,0.000000,0.000000
5788,L_p24_ROI,28,S4L,6,0.001216,0.003337
5789,L_p24_ROI,29,S4R,6,0.002945,0.007906
5790,L_p24_ROI,30,S5L,6,0.000213,0.001279


In [ ]:
M1 = T[(T.row_name=='M1L') | (T.row_name=='M1R')]
M1avg = (M1.groupby('cortical_roi', as_index=False)[['length', 'mean', 'max']]
           .mean())
M1avg['row_name'] = 'M1'

M1avg.sort_values('mean')[-10:]

,cortical_roi,length,mean,max,row_name
91,L_OFC_ROI,16.0,0.007693,0.078462,M1
34,L_6d_ROI,9.0,0.011490,0.084984,M1
87,L_MIP_ROI,8.0,0.021948,0.091034,M1
124,L_SCEF_ROI,11.0,0.019985,0.094257,M1
97,L_PEF_ROI,8.0,0.023815,0.120170,M1
81,L_LIPd_ROI,7.0,0.031093,0.120229,M1
62,L_FEF_ROI,8.0,0.028995,0.163633,M1
144,L_V1_ROI,46.0,0.024366,0.191555,M1
147,L_V3B_ROI,4.0,0.062767,0.200902,M1
27,L_4_ROI,52.0,0.010991,0.210296,M1
